# Computer Vision — Task 1: 8-Ball Pool Table Analysis

Work assembled by Alejandro Gonçalves (202205564), Francisca Mihalache (202206022), João Sousa (202205238), Rafael Pacheco (202206258).

---

**Objective:** Given images of an 8-ball pool table, this pipeline:
1. Detects and localizes all balls via bounding boxes
2. Identifies each ball's number based on color
3. Generates a top-view of the table
4. Exports results in JSON format + saves top-view images

**Allowed libraries:** OpenCV, NumPy, Matplotlib (+ tqdm, os, json, glob)

## Section 0 — Setup & Configuration

Import all required libraries and define configuration constants, helper functions, and data loading utilities.

In [ ]:
import cv2
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as patches
import os
import json
import glob
from tqdm import tqdm
from collections import Counter

# ---------------------------------------------------------------------------
# Configuration
# ---------------------------------------------------------------------------
DATA_DIR = "data"                  # Directory containing the input images
OUTPUT_DIR = "output"              # Directory for top-view images
RESULTS_FILE = "results.json"     # Output JSON file
os.makedirs(OUTPUT_DIR, exist_ok=True)

# Standard pool table aspect ratio (9-foot table: 254cm x 127cm ≈ 2:1)
TOP_VIEW_WIDTH = 800
TOP_VIEW_HEIGHT = 400

# Display settings
plt.rcParams['figure.figsize'] = (14, 8)
plt.rcParams['figure.dpi'] = 100

print("✓ All imports successful.")
print(f"  OpenCV version: {cv2.__version__}")
print(f"  NumPy version: {np.__version__}")

In [ ]:
# ---------------------------------------------------------------------------
# Helper Functions
# ---------------------------------------------------------------------------

def load_image(path):
    """Load an image in BGR and return it. Raises if file not found."""
    img = cv2.imread(path)
    if img is None:
        raise FileNotFoundError(f"Could not load image: {path}")
    return img

def bgr_to_rgb(img):
    """Convert BGR (OpenCV default) to RGB (matplotlib display)."""
    return cv2.cvtColor(img, cv2.COLOR_BGR2RGB)

def show_images(images, titles=None, figsize=(16, 6), cmap=None):
    """Display a list of images side by side."""
    n = len(images)
    fig, axes = plt.subplots(1, n, figsize=figsize)
    if n == 1:
        axes = [axes]
    for i, (ax, img) in enumerate(zip(axes, images)):
        if len(img.shape) == 2:  # Grayscale
            ax.imshow(img, cmap=cmap or 'gray')
        else:
            ax.imshow(bgr_to_rgb(img))
        if titles:
            ax.set_title(titles[i], fontsize=11)
        ax.axis('off')
    plt.tight_layout()
    plt.show()

def show_image(img, title="", figsize=(10, 7)):
    """Display a single image."""
    show_images([img], [title], figsize=figsize)

# Load all image paths
image_paths = sorted(glob.glob(os.path.join(DATA_DIR, "*.jpg")))
print(f"✓ Found {len(image_paths)} images in '{DATA_DIR}/'")

# Quick check — show the first image
sample_img = load_image(image_paths[0])
print(f"  Sample image shape: {sample_img.shape}")
show_image(sample_img, f"Sample: {os.path.basename(image_paths[0])}")

## Section 1 — HSV Color Exploration (Robustness Preparation)

Before jumping into ball identification, we explore the HSV color space across the dataset.
This helps us:
- Understand the range of possible ball colors under different lighting
- Establish robust HSV thresholds for table detection and ball classification

> **From project ideas:** "Antes de começar a parte 2, faz sentido ter uma filtragem dos intervalos de valores de intensidade para todos os parâmetros de cor HSV relativos às bolas, para termos noção do tipo de cores possíveis." 

In [ ]:
# ---------------------------------------------------------------------------
# 1.1 — Visualize HSV decomposition of a sample image
# ---------------------------------------------------------------------------
sample_hsv = cv2.cvtColor(sample_img, cv2.COLOR_BGR2HSV)

fig, axes = plt.subplots(1, 4, figsize=(20, 5))
axes[0].imshow(bgr_to_rgb(sample_img))
axes[0].set_title("Original (RGB)")
axes[1].imshow(sample_hsv[:, :, 0], cmap='hsv')
axes[1].set_title("Hue Channel")
axes[2].imshow(sample_hsv[:, :, 1], cmap='gray')
axes[2].set_title("Saturation Channel")
axes[3].imshow(sample_hsv[:, :, 2], cmap='gray')
axes[3].set_title("Value Channel")
for ax in axes:
    ax.axis('off')
plt.suptitle("HSV Decomposition", fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# ---------------------------------------------------------------------------
# 1.2 — Aggregate HSV statistics across all images
# ---------------------------------------------------------------------------
# We sample a small central region of each image to understand the table color
# and overall illumination conditions across the dataset.

hue_samples = []
sat_samples = []
val_samples = []

for path in tqdm(image_paths, desc="Sampling HSV statistics"):
    img = load_image(path)
    hsv = cv2.cvtColor(img, cv2.COLOR_BGR2HSV)
    h, w = hsv.shape[:2]
    # Sample central 50% of image (likely table region)
    cy, cx = h // 2, w // 2
    roi = hsv[cy - h//4 : cy + h//4, cx - w//4 : cx + w//4]
    hue_samples.extend(roi[:, :, 0].flatten().tolist())
    sat_samples.extend(roi[:, :, 1].flatten().tolist())
    val_samples.extend(roi[:, :, 2].flatten().tolist())

fig, axes = plt.subplots(1, 3, figsize=(18, 4))
axes[0].hist(hue_samples, bins=180, color='orange', alpha=0.7)
axes[0].set_title("Hue Distribution (center region)")
axes[0].set_xlabel("Hue (0-179)")
axes[1].hist(sat_samples, bins=256, color='green', alpha=0.7)
axes[1].set_title("Saturation Distribution")
axes[1].set_xlabel("Saturation (0-255)")
axes[2].hist(val_samples, bins=256, color='blue', alpha=0.7)
axes[2].set_title("Value Distribution")
axes[2].set_xlabel("Value (0-255)")
plt.suptitle("HSV Statistics Across Dataset (Central ROI)", fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

print(f"Hue — median: {int(np.median(hue_samples))}, mean: {np.mean(hue_samples):.1f}")
print(f"Sat — median: {int(np.median(sat_samples))}, mean: {np.mean(sat_samples):.1f}")
print(f"Val — median: {int(np.median(val_samples))}, mean: {np.mean(val_samples):.1f}")

## Section 2 — Part 1: Table Detection

The pool table has a distinctive **blue/green felt** surface. We detect it by:
1. Converting to HSV and thresholding on the table's hue range
2. Cleaning up the mask with morphological operations
3. Finding the largest contour → table boundary
4. Extracting the 4 corner points (needed later for perspective transform)

In [ ]:
# ---------------------------------------------------------------------------
# 2.1 — Table detection via HSV color thresholding
# ---------------------------------------------------------------------------

def detect_table_mask(img, debug=False):
    """
    Detect the pool table felt using HSV color thresholding.
    Returns a binary mask of the table region.
    """
    hsv = cv2.cvtColor(img, cv2.COLOR_BGR2HSV)
    
    # Blue/green felt — typical HSV ranges for pool table cloth
    # Hue: ~85-130 (blue-green), Sat: >40 (colored, not gray), Val: >40 (not too dark)
    lower_table = np.array([85, 40, 40])
    upper_table = np.array([135, 255, 255])
    mask = cv2.inRange(hsv, lower_table, upper_table)
    
    # Morphological cleanup
    kernel = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (15, 15))
    mask = cv2.morphologyEx(mask, cv2.MORPH_CLOSE, kernel, iterations=3)
    mask = cv2.morphologyEx(mask, cv2.MORPH_OPEN, kernel, iterations=2)
    
    if debug:
        show_images([img, mask], ["Original", "Table Mask"], cmap='gray')
    
    return mask


def get_table_contour(mask):
    """
    Find the largest contour in the table mask — assumed to be the table surface.
    Returns the contour (or None if not found).
    """
    contours, _ = cv2.findContours(mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    if not contours:
        return None
    # The largest contour by area is the table
    table_contour = max(contours, key=cv2.contourArea)
    return table_contour


def get_table_corners(contour, img_shape):
    """
    Extract the 4 corner points of the table from its contour.
    Uses approxPolyDP to simplify, then picks the 4 extreme corners.
    Returns corners in order: [top-left, top-right, bottom-right, bottom-left].
    """
    # Try polygon approximation first
    epsilon = 0.02 * cv2.arcLength(contour, True)
    approx = cv2.approxPolyDP(contour, epsilon, True)
    
    if len(approx) == 4:
        pts = approx.reshape(4, 2).astype(np.float32)
    else:
        # Fallback: use the bounding rectangle corners from the convex hull
        hull = cv2.convexHull(contour)
        rect = cv2.minAreaRect(hull)
        pts = cv2.boxPoints(rect).astype(np.float32)
    
    # Order points: top-left, top-right, bottom-right, bottom-left
    pts = order_points(pts)
    return pts


def order_points(pts):
    """
    Order 4 points as: [top-left, top-right, bottom-right, bottom-left].
    """
    rect = np.zeros((4, 2), dtype=np.float32)
    # Sum: top-left has smallest sum, bottom-right has largest
    s = pts.sum(axis=1)
    rect[0] = pts[np.argmin(s)]
    rect[2] = pts[np.argmax(s)]
    # Difference: top-right has smallest diff, bottom-left has largest
    d = np.diff(pts, axis=1)
    rect[1] = pts[np.argmin(d)]
    rect[3] = pts[np.argmax(d)]
    return rect

# Test on sample image
table_mask = detect_table_mask(sample_img, debug=True)
table_contour = get_table_contour(table_mask)
print(f"Table contour area: {cv2.contourArea(table_contour):.0f} pixels")

In [ ]:
# ---------------------------------------------------------------------------
# 2.2 — Visualize detected table boundary and corners
# ---------------------------------------------------------------------------
corners = get_table_corners(table_contour, sample_img.shape)

vis = sample_img.copy()
cv2.drawContours(vis, [table_contour], -1, (0, 255, 0), 3)

# Draw corners
corner_labels = ["TL", "TR", "BR", "BL"]
for i, (pt, label) in enumerate(zip(corners, corner_labels)):
    x, y = int(pt[0]), int(pt[1])
    cv2.circle(vis, (x, y), 10, (0, 0, 255), -1)
    cv2.putText(vis, label, (x + 12, y - 5), cv2.FONT_HERSHEY_SIMPLEX, 0.7, (0, 0, 255), 2)

show_image(vis, "Detected Table Boundary & Corners")
print(f"Corners (TL, TR, BR, BL):")
for label, pt in zip(corner_labels, corners):
    print(f"  {label}: ({pt[0]:.0f}, {pt[1]:.0f})")

## Section 3 — Part 1: Ball Detection by Shape

Balls are always circular. We detect them using:
1. **Pre-processing:** Gaussian blur + contrast enhancement (CLAHE) within the table region
2. **Circle detection:** `cv2.HoughCircles` for robust circle finding
3. **Validation:** reject detections outside the table mask or near pocket regions

> **From project ideas:** "Primeiro fazer segmentação por shape (sempre circular, com uma threshold específica)" and "regular contrastes e sharpness para tornar essa detection mais robusta." 

In [ ]:
# ---------------------------------------------------------------------------
# 3.1 — Ball detection using HoughCircles
# ---------------------------------------------------------------------------

def detect_balls(img, table_mask, debug=False):
    """
    Detect pool balls using Hough Circle Transform.
    Returns list of (x, y, radius) tuples for detected circles.
    """
    h, w = img.shape[:2]
    
    # Apply table mask — set non-table pixels to black
    masked = cv2.bitwise_and(img, img, mask=table_mask)
    
    # Convert to grayscale
    gray = cv2.cvtColor(masked, cv2.COLOR_BGR2GRAY)
    
    # Apply CLAHE for contrast enhancement
    clahe = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8, 8))
    gray = clahe.apply(gray)
    
    # Gaussian blur to reduce noise
    gray = cv2.GaussianBlur(gray, (9, 9), 2)
    
    # Estimate ball size based on image dimensions
    # Pool balls occupy roughly 2-4% of table width
    min_radius = int(w * 0.012)
    max_radius = int(w * 0.035)
    
    # Hough Circle Transform
    circles = cv2.HoughCircles(
        gray,
        cv2.HOUGH_GRADIENT,
        dp=1.2,
        minDist=min_radius * 2,      # Balls can't overlap
        param1=80,                    # Upper Canny threshold
        param2=30,                    # Accumulator threshold — lower = more detections
        minRadius=min_radius,
        maxRadius=max_radius,
    )
    
    if circles is None:
        return []
    
    circles = np.round(circles[0, :]).astype(int)
    
    # Filter: keep only circles whose center is on the table
    valid_circles = []
    for (x, y, r) in circles:
        if 0 <= y < h and 0 <= x < w and table_mask[y, x] > 0:
            valid_circles.append((x, y, r))
    
    if debug:
        vis = img.copy()
        for (x, y, r) in valid_circles:
            cv2.circle(vis, (x, y), r, (0, 255, 0), 2)
            cv2.circle(vis, (x, y), 2, (0, 0, 255), 3)
        show_images(
            [gray, vis],
            [f"Preprocessed (CLAHE + Blur)", f"Detected: {len(valid_circles)} balls"],
            figsize=(16, 7)
        )
    
    return valid_circles

# Test on sample image
balls = detect_balls(sample_img, table_mask, debug=True)
print(f"Detected {len(balls)} balls in sample image.")

In [ ]:
# ---------------------------------------------------------------------------
# 3.2 — Test detection across multiple images to validate robustness
# ---------------------------------------------------------------------------
detection_counts = []

fig, axes = plt.subplots(2, 4, figsize=(20, 10))
axes = axes.flatten()

for i, path in enumerate(image_paths[:8]):
    img = load_image(path)
    mask = detect_table_mask(img)
    detected = detect_balls(img, mask)
    detection_counts.append(len(detected))
    
    vis = img.copy()
    for (x, y, r) in detected:
        cv2.circle(vis, (x, y), r, (0, 255, 0), 2)
        cv2.circle(vis, (x, y), 2, (0, 0, 255), 3)
    
    axes[i].imshow(bgr_to_rgb(vis))
    axes[i].set_title(f"{os.path.basename(path)[:12]}... — {len(detected)} balls", fontsize=9)
    axes[i].axis('off')

plt.suptitle("Ball Detection on First 8 Images", fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

print(f"Detection counts: {detection_counts}")
print(f"Average: {np.mean(detection_counts):.1f}, Min: {min(detection_counts)}, Max: {max(detection_counts)}")

## Section 4 — Part 1: Bounding Boxes

Convert each detected circle into a **square bounding box** and produce normalized coordinates (matching the required JSON output format: xmin, xmax, ymin, ymax as fractions of image width/height).

In [ ]:
# ---------------------------------------------------------------------------
# 4.1 — Convert circles to bounding boxes
# ---------------------------------------------------------------------------

def circles_to_bboxes(circles, img_shape):
    """
    Convert (x, y, radius) circles to bounding boxes.
    Returns list of dicts with normalized coordinates:
    {xmin, xmax, ymin, ymax} in range [0, 1].
    """
    h, w = img_shape[:2]
    bboxes = []
    for (cx, cy, r) in circles:
        # Add a small padding around the circle
        pad = int(r * 0.15)
        x1 = max(0, cx - r - pad)
        y1 = max(0, cy - r - pad)
        x2 = min(w, cx + r + pad)
        y2 = min(h, cy + r + pad)
        
        bboxes.append({
            "xmin": x1 / w,
            "xmax": x2 / w,
            "ymin": y1 / h,
            "ymax": y2 / h,
            # Keep pixel coords for internal use
            "_x1": x1, "_y1": y1, "_x2": x2, "_y2": y2,
            "_cx": cx, "_cy": cy, "_r": r
        })
    return bboxes

# Test
bboxes = circles_to_bboxes(balls, sample_img.shape)

# Visualize bounding boxes
fig, ax = plt.subplots(1, 1, figsize=(14, 9))
ax.imshow(bgr_to_rgb(sample_img))
for bb in bboxes:
    rect = patches.Rectangle(
        (bb["_x1"], bb["_y1"]),
        bb["_x2"] - bb["_x1"],
        bb["_y2"] - bb["_y1"],
        linewidth=2, edgecolor='lime', facecolor='none'
    )
    ax.add_patch(rect)
ax.set_title(f"Bounding Boxes — {len(bboxes)} balls detected", fontsize=13)
ax.axis('off')
plt.tight_layout()
plt.show()

print(f"Sample bbox (normalized): xmin={bboxes[0]['xmin']:.4f}, xmax={bboxes[0]['xmax']:.4f}, "
      f"ymin={bboxes[0]['ymin']:.4f}, ymax={bboxes[0]['ymax']:.4f}")

## Section 5 — Part 2: Color Normalization

Before classifying ball colors, we normalize each ball ROI to reduce the effect of varying illumination. Steps:
1. Convert each ball ROI to HSV
2. Apply CLAHE on the Value channel (brightness equalization)
3. Optional saturation boost for better color discrimination

> **From project ideas:** "Perform contrast+saturation balance em HSV (para termos mais controlo de saturação, brilho, ângulos de luz e etc)" 

In [ ]:
# ---------------------------------------------------------------------------
# 5.1 — Ball ROI extraction and color normalization
# ---------------------------------------------------------------------------

def extract_ball_roi(img, bbox):
    """Extract the ball ROI from the image using the bounding box."""
    return img[bbox["_y1"]:bbox["_y2"], bbox["_x1"]:bbox["_x2"]].copy()

def create_circular_mask(roi):
    """Create a circular mask to focus only on the ball (ignore corners of bbox)."""
    h, w = roi.shape[:2]
    mask = np.zeros((h, w), dtype=np.uint8)
    cx, cy = w // 2, h // 2
    r = min(cx, cy) - 1
    cv2.circle(mask, (cx, cy), max(r, 1), 255, -1)
    return mask

def normalize_ball_color(roi):
    """
    Normalize the ball ROI for more consistent color classification.
    - CLAHE on V channel for brightness
    - Mild saturation boost
    """
    hsv = cv2.cvtColor(roi, cv2.COLOR_BGR2HSV)
    h, s, v = cv2.split(hsv)
    
    # CLAHE on value channel
    clahe = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(4, 4))
    v = clahe.apply(v)
    
    # Mild saturation boost (clamp to 255)
    s = np.clip(s.astype(np.int16) + 20, 0, 255).astype(np.uint8)
    
    hsv_norm = cv2.merge([h, s, v])
    return cv2.cvtColor(hsv_norm, cv2.COLOR_HSV2BGR), hsv_norm

# Visualize ROIs for the sample image
fig, axes = plt.subplots(2, min(len(bboxes), 8), figsize=(20, 5))
n_show = min(len(bboxes), 8)
for i in range(n_show):
    roi = extract_ball_roi(sample_img, bboxes[i])
    roi_norm, _ = normalize_ball_color(roi)
    if n_show > 1:
        axes[0, i].imshow(bgr_to_rgb(roi))
        axes[0, i].set_title(f"Ball {i+1}", fontsize=9)
        axes[0, i].axis('off')
        axes[1, i].imshow(bgr_to_rgb(roi_norm))
        axes[1, i].set_title(f"Normalized", fontsize=9)
        axes[1, i].axis('off')

if n_show > 1:
    axes[0, 0].set_ylabel("Original", fontsize=11)
    axes[1, 0].set_ylabel("Normalized", fontsize=11)
plt.suptitle("Ball ROI Extraction & Color Normalization", fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

## Section 6 — Part 2: Ball Number Identification by Color

We classify each ball by its dominant color using HSV analysis. The 8-ball pool color mapping:

| Ball # | Color     | Type   | Ball # | Color    | Type   |
|--------|-----------|--------|--------|----------|--------|
| 0      | White     | Cue    | 8      | Black    | 8-ball |
| 1      | Yellow    | Solid  | 9      | Yellow   | Stripe |
| 2      | Blue      | Solid  | 10     | Blue     | Stripe |
| 3      | Red       | Solid  | 11     | Red      | Stripe |
| 4      | Purple    | Solid  | 12     | Purple   | Stripe |
| 5      | Orange    | Solid  | 13     | Orange   | Stripe |
| 6      | Green     | Solid  | 14     | Green    | Stripe |
| 7      | Maroon    | Solid  | 15     | Maroon   | Stripe |

**Strategy:**
1. Determine the dominant color category (yellow, blue, red, etc.)
2. Compute white-pixel ratio to distinguish solids vs stripes
3. Special handling for cue ball (all white) and 8-ball (all black)

> **From project ideas:** "para distinguir entre bolas com as mesmas combinações de cores, podemos ver o ratio de branco e vermelho nas bolas e só aí atribuir pontuação" 

In [ ]:
# ---------------------------------------------------------------------------
# 6.1 — Color classification engine
# ---------------------------------------------------------------------------

# HSV color ranges for each ball color category
# Format: (lower_H, upper_H, lower_S, lower_V, label)
# Note: OpenCV uses H: 0-179, S: 0-255, V: 0-255
BALL_COLOR_RANGES = {
    "yellow":  {"h_range": (20, 35),   "s_min": 80,  "v_min": 80},
    "orange":  {"h_range": (10, 22),   "s_min": 100, "v_min": 80},
    "red":     {"h_range": [(0, 10), (170, 179)], "s_min": 80, "v_min": 60},  # Red wraps
    "maroon":  {"h_range": [(0, 12), (165, 179)], "s_min": 40, "v_min": 30, "v_max": 120},
    "blue":    {"h_range": (100, 130), "s_min": 80,  "v_min": 50},
    "purple":  {"h_range": (125, 155), "s_min": 40,  "v_min": 40},
    "green":   {"h_range": (35, 85),   "s_min": 60,  "v_min": 40},
}

# Solid/stripe pairs: color_name -> (solid_number, stripe_number)
COLOR_TO_NUMBERS = {
    "yellow":  (1, 9),
    "blue":    (2, 10),
    "red":     (3, 11),
    "purple":  (4, 12),
    "orange":  (5, 13),
    "green":   (6, 14),
    "maroon":  (7, 15),
}


def compute_color_ratios(roi_hsv, mask):
    """
    Compute the ratio of white, black, and colored pixels in the ball ROI.
    Returns dict with pixel ratios.
    """
    h, s, v = cv2.split(roi_hsv)
    
    total_pixels = max(cv2.countNonZero(mask), 1)
    
    # White pixels: low saturation, high value
    white_mask = cv2.bitwise_and(
        cv2.inRange(s, 0, 50),
        cv2.inRange(v, 170, 255)
    )
    white_mask = cv2.bitwise_and(white_mask, mask)
    white_ratio = cv2.countNonZero(white_mask) / total_pixels
    
    # Black pixels: very low value
    black_mask = cv2.inRange(v, 0, 50)
    black_mask = cv2.bitwise_and(black_mask, mask)
    black_ratio = cv2.countNonZero(black_mask) / total_pixels
    
    # Colored pixels: moderate-to-high saturation
    color_mask = cv2.inRange(s, 60, 255)
    color_mask = cv2.bitwise_and(color_mask, mask)
    color_ratio = cv2.countNonZero(color_mask) / total_pixels
    
    return {
        "white": white_ratio,
        "black": black_ratio,
        "colored": color_ratio,
    }


def classify_ball_color(roi_hsv, mask):
    """
    Determine the dominant color category of a ball ROI.
    Returns the color name string.
    """
    h, s, v = cv2.split(roi_hsv)
    
    # Only consider pixels inside the mask
    best_color = "unknown"
    best_count = 0
    
    for color_name, params in BALL_COLOR_RANGES.items():
        h_range = params["h_range"]
        s_min = params["s_min"]
        v_min = params["v_min"]
        v_max = params.get("v_max", 255)
        
        # Handle colors with multiple hue ranges (red, maroon)
        if isinstance(h_range, list):
            color_mask = np.zeros_like(h, dtype=np.uint8)
            for (h_lo, h_hi) in h_range:
                partial = cv2.inRange(roi_hsv, 
                    np.array([h_lo, s_min, v_min]), 
                    np.array([h_hi, 255, v_max]))
                color_mask = cv2.bitwise_or(color_mask, partial)
        else:
            h_lo, h_hi = h_range
            color_mask = cv2.inRange(roi_hsv,
                np.array([h_lo, s_min, v_min]),
                np.array([h_hi, 255, v_max]))
        
        color_mask = cv2.bitwise_and(color_mask, mask)
        count = cv2.countNonZero(color_mask)
        
        if count > best_count:
            best_count = count
            best_color = color_name
    
    return best_color


def identify_ball_number(roi_bgr):
    """
    Full ball identification pipeline:
    1. Normalize colors
    2. Classify dominant color
    3. Determine solid vs stripe via white ratio
    4. Return ball number
    """
    _, roi_hsv = normalize_ball_color(roi_bgr)
    circ_mask = create_circular_mask(roi_bgr)
    
    # Compute color ratios
    ratios = compute_color_ratios(roi_hsv, circ_mask)
    
    # Special case: cue ball (mostly white)
    if ratios["white"] > 0.55:
        return 0, "white", ratios
    
    # Special case: 8-ball (mostly black)
    if ratios["black"] > 0.45 and ratios["colored"] < 0.15:
        return 8, "black", ratios
    
    # Classify dominant color
    color = classify_ball_color(roi_hsv, circ_mask)
    
    if color == "unknown":
        return -1, "unknown", ratios
    
    # Solid vs stripe: stripes have significantly more white
    solid_num, stripe_num = COLOR_TO_NUMBERS[color]
    
    # Stripes have a white band — typically white_ratio > 0.25
    if ratios["white"] > 0.25:
        return stripe_num, color, ratios
    else:
        return solid_num, color, ratios


print("✓ Ball identification functions defined.")

In [ ]:
# ---------------------------------------------------------------------------
# 6.2 — Test ball identification on sample image
# ---------------------------------------------------------------------------
fig, axes = plt.subplots(2, min(len(bboxes), 8), figsize=(20, 6))
n_show = min(len(bboxes), 8)

for i in range(n_show):
    roi = extract_ball_roi(sample_img, bboxes[i])
    ball_num, color, ratios = identify_ball_number(roi)
    
    axes[0, i].imshow(bgr_to_rgb(roi))
    axes[0, i].set_title(f"#{ball_num} ({color})", fontsize=9, fontweight='bold')
    axes[0, i].axis('off')
    
    # Show the color distribution
    labels = list(ratios.keys())
    values = list(ratios.values())
    colors_bar = ['whitesmoke', 'black', 'coral']
    axes[1, i].barh(labels, values, color=colors_bar)
    axes[1, i].set_xlim(0, 1)
    axes[1, i].set_title("Color Ratios", fontsize=8)

plt.suptitle("Ball Number Identification — Sample Image", fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

## Section 7 — Part 2: Robustness Checks

Validation steps to ensure detection quality:
1. **Duplicate check:** Flag images where two balls are assigned the same number
2. **White & black ball check:** Soft flag whether both balls #0 (cue) and #8 are present
3. **Confidence warnings:** Log uncertain classifications

> **From project ideas:** "Não podem existir duas bolas com o mesmo número (imagem deve ser flagged como 'Suspicious')" 

In [ ]:
# ---------------------------------------------------------------------------
# 7.1 — Robustness validation functions
# ---------------------------------------------------------------------------

def validate_detections(ball_results, image_name=""):
    """
    Validate the ball detection results for an image.
    Returns a list of warning strings.
    """
    warnings = []
    
    numbers = [b["number"] for b in ball_results if b["number"] >= 0]
    
    # Check 1: Duplicate ball numbers
    counts = Counter(numbers)
    duplicates = {num: cnt for num, cnt in counts.items() if cnt > 1}
    if duplicates:
        warnings.append(f"⚠️  SUSPICIOUS — Duplicate ball numbers detected: {duplicates}")
    
    # Check 2: Cue ball (0) and 8-ball present?
    has_cue = 0 in numbers
    has_eight = 8 in numbers
    if not has_cue:
        warnings.append(f"ℹ️  Note: Cue ball (#0) not detected")
    if not has_eight:
        warnings.append(f"ℹ️  Note: 8-ball (#8) not detected")
    
    # Check 3: Unknown balls
    unknowns = sum(1 for b in ball_results if b["number"] == -1)
    if unknowns > 0:
        warnings.append(f"⚠️  {unknowns} ball(s) could not be identified (marked as -1)")
    
    # Check 4: Reasonable ball count (pool has max 16 balls: cue + 15 object balls)
    if len(ball_results) > 16:
        warnings.append(f"⚠️  SUSPICIOUS — More than 16 balls detected ({len(ball_results)})")
    
    if warnings:
        prefix = f"[{image_name}] " if image_name else ""
        for w in warnings:
            print(f"  {prefix}{w}")
    
    return warnings

# Quick test on our sample results
sample_results = []
for bb in bboxes:
    roi = extract_ball_roi(sample_img, bb)
    num, color, ratios = identify_ball_number(roi)
    sample_results.append({"number": num, "color": color, "ratios": ratios})

print(f"Ball numbers found: {[r['number'] for r in sample_results]}")
print()
validate_detections(sample_results, os.path.basename(image_paths[0]))

## Section 8 — Part 3: Top-View Perspective Transformation

Generate a rectified top-down view of the pool table by:
1. Using the table corners detected in Section 2
2. Defining destination points for a standard rectangle
3. Computing and applying the perspective transform
4. Optionally mapping ball positions to the new coordinate system

> **From project ideas:** "Obter os cantos azuis novamente, e aplicar transformações para que os cantos tenham posições do género: linha reta entre canto superior e inferior, linha reta entre canto esquerdo e direito" 

In [ ]:
# ---------------------------------------------------------------------------
# 8.1 — Perspective transformation to top view
# ---------------------------------------------------------------------------

def generate_top_view(img, table_corners, width=TOP_VIEW_WIDTH, height=TOP_VIEW_HEIGHT):
    """
    Warp the pool table region to a rectangular top-down view.
    
    Args:
        img: Original image (BGR)
        table_corners: 4 corner points [TL, TR, BR, BL] in the original image
        width, height: Dimensions of the output top-view image
    
    Returns:
        top_view: The rectified top-view image
        M: The perspective transformation matrix
    """
    src_pts = table_corners.astype(np.float32)
    dst_pts = np.array([
        [0, 0],
        [width - 1, 0],
        [width - 1, height - 1],
        [0, height - 1]
    ], dtype=np.float32)
    
    M = cv2.getPerspectiveTransform(src_pts, dst_pts)
    top_view = cv2.warpPerspective(img, M, (width, height))
    
    return top_view, M


def transform_point(point, M):
    """Transform a single (x, y) point using perspective matrix M."""
    pt = np.array([[[point[0], point[1]]]], dtype=np.float32)
    transformed = cv2.perspectiveTransform(pt, M)
    return transformed[0][0]

# Generate top-view of sample image
top_view, M = generate_top_view(sample_img, corners)

show_images(
    [sample_img, top_view],
    ["Original Image", "Top-View Transformation"],
    figsize=(18, 7)
)

In [ ]:
# ---------------------------------------------------------------------------
# 8.2 — Top-view with ball positions mapped
# ---------------------------------------------------------------------------
top_view_annotated = top_view.copy()

for bb in bboxes:
    # Transform ball center to top-view coordinates
    cx, cy = bb["_cx"], bb["_cy"]
    new_pt = transform_point((cx, cy), M)
    nx, ny = int(new_pt[0]), int(new_pt[1])
    
    # Only draw if the point is within the top-view bounds
    if 0 <= nx < TOP_VIEW_WIDTH and 0 <= ny < TOP_VIEW_HEIGHT:
        cv2.circle(top_view_annotated, (nx, ny), 10, (0, 255, 255), -1)
        cv2.circle(top_view_annotated, (nx, ny), 10, (0, 0, 0), 2)

show_images(
    [top_view, top_view_annotated],
    ["Top-View (Clean)", "Top-View with Ball Positions"],
    figsize=(18, 6)
)

In [ ]:
# ---------------------------------------------------------------------------
# 8.3 — Test top-view on multiple images
# ---------------------------------------------------------------------------
fig, axes = plt.subplots(2, 4, figsize=(20, 10))

for i, path in enumerate(image_paths[:4]):
    img = load_image(path)
    mask = detect_table_mask(img)
    contour = get_table_contour(mask)
    
    if contour is not None:
        crns = get_table_corners(contour, img.shape)
        tv, _ = generate_top_view(img, crns)
    else:
        tv = np.zeros((TOP_VIEW_HEIGHT, TOP_VIEW_WIDTH, 3), dtype=np.uint8)
    
    axes[0, i].imshow(bgr_to_rgb(img))
    axes[0, i].set_title(f"Original", fontsize=10)
    axes[0, i].axis('off')
    axes[1, i].imshow(bgr_to_rgb(tv))
    axes[1, i].set_title(f"Top View", fontsize=10)
    axes[1, i].axis('off')

plt.suptitle("Top-View Transformation — Multiple Images", fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

## Section 9 — Full Pipeline & JSON Output

Wrap all previous steps into a single pipeline function and batch-process all images.

**JSON output format** (matching `output_example.json`):
```json
[
  {
    "image_path": "data/image_name.jpg",
    "num_balls": 15,
    "balls": [
      {"number": 13, "xmin": 0.397, "xmax": 0.416, "ymin": 0.257, "ymax": 0.288},
      ...
    ]
  }
]
```

**Additional output:** Top-view images saved to `output/<image_name>.jpg`

In [ ]:
# ---------------------------------------------------------------------------
# 9.1 — Complete pipeline function
# ---------------------------------------------------------------------------

def process_image(image_path, save_top_view=True, verbose=False):
    """
    Full processing pipeline for a single image.
    
    Returns:
        result: dict with image_path, num_balls, balls (with number and bbox)
        warnings: list of validation warnings
    """
    img = load_image(image_path)
    h, w = img.shape[:2]
    image_name = os.path.basename(image_path)
    
    # Step 1: Detect table
    table_mask = detect_table_mask(img)
    table_contour = get_table_contour(table_mask)
    
    if table_contour is None:
        print(f"  ⚠️  [{image_name}] Could not detect table!")
        return {"image_path": image_path, "num_balls": 0, "balls": []}, ["Table not detected"]
    
    # Step 2: Detect balls
    circles = detect_balls(img, table_mask)
    bboxes = circles_to_bboxes(circles, img.shape)
    
    # Step 3: Identify ball numbers
    balls_output = []
    for bb in bboxes:
        roi = extract_ball_roi(img, bb)
        if roi.size == 0:
            continue
        num, color, ratios = identify_ball_number(roi)
        balls_output.append({
            "number": num,
            "xmin": round(bb["xmin"], 6),
            "xmax": round(bb["xmax"], 6),
            "ymin": round(bb["ymin"], 6),
            "ymax": round(bb["ymax"], 6),
        })
    
    # Step 4: Validation
    warnings = validate_detections(balls_output, image_name)
    
    # Step 5: Generate and save top-view
    if save_top_view:
        corners = get_table_corners(table_contour, img.shape)
        top_view, _ = generate_top_view(img, corners)
        out_path = os.path.join(OUTPUT_DIR, image_name)
        cv2.imwrite(out_path, top_view)
        if verbose:
            print(f"  ✓ Top-view saved: {out_path}")
    
    result = {
        "image_path": image_path,
        "num_balls": len(balls_output),
        "balls": balls_output,
    }
    
    return result, warnings


# Test on a single image
test_result, test_warnings = process_image(image_paths[0], verbose=True)
print(f"\nResult for {test_result['image_path']}:")
print(f"  Balls detected: {test_result['num_balls']}")
print(f"  Ball numbers: {[b['number'] for b in test_result['balls']]}")

In [ ]:
# ---------------------------------------------------------------------------
# 9.2 — Batch process all images
# ---------------------------------------------------------------------------
all_results = []
all_warnings = {}

print("=" * 60)
print("Running full pipeline on all images...")
print("=" * 60)

for path in tqdm(image_paths, desc="Processing images"):
    result, warnings = process_image(path, save_top_view=True)
    all_results.append(result)
    if warnings:
        all_warnings[os.path.basename(path)] = warnings

# Save JSON results
with open(RESULTS_FILE, "w") as f:
    json.dump(all_results, f, indent=4)

print(f"\n✓ Results saved to '{RESULTS_FILE}'")
print(f"✓ Top-view images saved to '{OUTPUT_DIR}/'")
print(f"✓ Processed {len(all_results)} images")
print(f"  Images with warnings: {len(all_warnings)}")

In [ ]:
# ---------------------------------------------------------------------------
# 9.3 — Summary statistics
# ---------------------------------------------------------------------------
ball_counts = [r["num_balls"] for r in all_results]
all_numbers = []
for r in all_results:
    all_numbers.extend([b["number"] for b in r["balls"]])

print("=== Detection Summary ===")
print(f"Total images processed: {len(all_results)}")
print(f"Total balls detected: {sum(ball_counts)}")
print(f"Avg balls per image: {np.mean(ball_counts):.1f}")
print(f"Min / Max balls: {min(ball_counts)} / {max(ball_counts)}")
print()

# Distribution of ball numbers
num_counter = Counter(all_numbers)
print("Ball number distribution across all images:")
for num in sorted(num_counter.keys()):
    print(f"  Ball #{num}: found {num_counter[num]} times")

# Warnings summary
if all_warnings:
    print(f"\n=== Warnings ({len(all_warnings)} images) ===")
    for img_name, warns in all_warnings.items():
        for w in warns:
            print(f"  [{img_name}] {w}")

## Section 10 — Results Visualization

Visual summary of the pipeline results across the dataset.

In [ ]:
# ---------------------------------------------------------------------------
# 10.1 — Grid visualization: Original + Detections + Top-View
# ---------------------------------------------------------------------------
n_display = 6
fig, axes = plt.subplots(3, n_display, figsize=(24, 14))

for i, path in enumerate(image_paths[:n_display]):
    img = load_image(path)
    name = os.path.basename(path)
    
    # Get detections
    mask = detect_table_mask(img)
    contour = get_table_contour(mask)
    circles = detect_balls(img, mask)
    bbs = circles_to_bboxes(circles, img.shape)
    
    # Original
    axes[0, i].imshow(bgr_to_rgb(img))
    axes[0, i].set_title(f"{name[:15]}...", fontsize=8)
    axes[0, i].axis('off')
    
    # Detections
    vis = img.copy()
    for bb in bbs:
        roi = extract_ball_roi(img, bb)
        if roi.size == 0:
            continue
        num, color, _ = identify_ball_number(roi)
        cv2.rectangle(vis, (bb["_x1"], bb["_y1"]), (bb["_x2"], bb["_y2"]), (0, 255, 0), 2)
        cv2.putText(vis, str(num), (bb["_x1"], bb["_y1"] - 5),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.5, (0, 255, 255), 2)
    axes[1, i].imshow(bgr_to_rgb(vis))
    axes[1, i].set_title(f"{len(bbs)} balls", fontsize=9)
    axes[1, i].axis('off')
    
    # Top-view
    if contour is not None:
        crns = get_table_corners(contour, img.shape)
        tv, _ = generate_top_view(img, crns)
        axes[2, i].imshow(bgr_to_rgb(tv))
    axes[2, i].set_title("Top View", fontsize=9)
    axes[2, i].axis('off')

axes[0, 0].set_ylabel("Original", fontsize=12, fontweight='bold')
axes[1, 0].set_ylabel("Detected", fontsize=12, fontweight='bold')
axes[2, 0].set_ylabel("Top View", fontsize=12, fontweight='bold')

plt.suptitle("Pipeline Results — Sample Images", fontsize=15, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# ---------------------------------------------------------------------------
# 10.2 — Detection count histogram
# ---------------------------------------------------------------------------
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Ball count distribution
axes[0].hist(ball_counts, bins=range(0, 20), color='steelblue', edgecolor='white', alpha=0.8)
axes[0].set_xlabel("Number of Balls Detected")
axes[0].set_ylabel("Number of Images")
axes[0].set_title("Ball Count Distribution Across Dataset")
axes[0].axvline(np.mean(ball_counts), color='red', linestyle='--', label=f'Mean: {np.mean(ball_counts):.1f}')
axes[0].legend()

# Ball number distribution
nums_sorted = sorted(num_counter.keys())
axes[1].bar(
    [str(n) for n in nums_sorted],
    [num_counter[n] for n in nums_sorted],
    color='coral', edgecolor='white', alpha=0.8
)
axes[1].set_xlabel("Ball Number")
axes[1].set_ylabel("Times Detected")
axes[1].set_title("Ball Number Frequency Across Dataset")

plt.tight_layout()
plt.show()

In [ ]:
# ---------------------------------------------------------------------------
# 10.3 — Show sample JSON output
# ---------------------------------------------------------------------------
print("Sample JSON output (first image):")
print(json.dumps(all_results[0], indent=2))